# Tuning the model

## Leraning Rate

In [ ]:
from src.cnn_utils import get_dataset

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

In [ ]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import wandb


# =========================================================
# MODEL
# =========================================================
# A simple convolutional block: Conv2d -> BatchNorm -> ReLU -> (optional MaxPool)

class ConvBlock(nn.Module):
    """Conv2d -> BatchNorm2d -> ReLU -> optional MaxPool2d"""
    def __init__(self, in_channels, out_channels, use_pool=False):
        super().__init__()

        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]

        if use_pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

# constructing CNN with depth 12 from previous runs
class DepthCNN(nn.Module):
    def __init__(
        self,
        depth=12,
        in_channels=3,
        num_classes=10,
        base_channels=32,
        max_channels=256,
        dropout=0.5,
        inputsize=224,
    ):
        super().__init__()

        layers = []
        current_in = in_channels
        current_out = base_channels
        current_size = inputsize
        pool_count = 0

        for i in range(depth):
            want_pool = ((i + 1) % 2 == 0)
            use_pool = want_pool and current_size >= 2 and pool_count < 4

            layers.append(ConvBlock(current_in, current_out, use_pool))

            current_in = current_out

            if use_pool:
                current_size //= 2
                pool_count += 1
                current_out = min(current_out * 2, max_channels)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(current_in, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================================================
# TRAIN / EVAL
# =========================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train_model()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time()

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_time


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


# =========================================================
# SETTINGS
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

depth = 12 # depth evaluated from previous runs
epochs = 50 # since in previous runs the model converged at around 30 epochs, we can reduce the number of epochs for tuning to save time
batch_size = 64 # default value
weight_decay = 1e-4 # default value

# try different learning rates (some values are absurd)
lrs = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)


# =========================================================
# LR TUNING LOOP
# =========================================================

results = {}

for lr in lrs:
    print("\n" + "=" * 80)
    print(f"Training with lr = {lr}")
    print("=" * 80)

    model = DepthCNN(depth=depth).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        nesterov=True,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_val_loss = float("inf")
    best_train_loss = float("inf")
    best_epoch = -1

    run = wandb.init(
        project="MPW-CNN",
        entity="MSE_DeLearn_SPR26",
        name=f"lr_tuning_MomentumNesterov_depth12_lr{lr}",
        config={
            "depth": depth,
            "lr": lr,
            "optimizer": "SGD_Nesterov",
            "momentum": 0.9,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "batch_size": batch_size,
        },
        reinit=True
    )

    for epoch in range(epochs):
        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1

        if train_acc > best_train_acc:
            best_train_acc = train_acc

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        if train_loss < best_train_loss:
            best_train_loss = train_loss

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_acc": best_val_acc,
            "best_train_acc": best_train_acc,
            "best_val_loss": best_val_loss,
            "best_train_loss": best_train_loss,
            "epoch_time_sec": epoch_time,
        })

    wandb.summary["best_val_acc"] = best_val_acc
    wandb.summary["best_train_acc"] = best_train_acc
    wandb.summary["best_val_loss"] = best_val_loss
    wandb.summary["best_train_loss"] = best_train_loss
    wandb.summary["best_epoch"] = best_epoch
    wandb.finish()

    results[lr] = {
        "lr": lr,
        "best_val_acc": best_val_acc,
        "best_train_acc": best_train_acc,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_train_loss": best_train_loss,
    }


# =========================================================
# SUMMARY
# =========================================================

print("\n" + "=" * 80)
print("LR RESULTS")
print("=" * 80)

for lr, result in results.items():
    print(
        f"lr={lr:<8} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_train_acc={result['best_train_acc']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_train_loss={result['best_train_loss']:.4f}"
    )

best_lr = max(results, key=lambda lr: results[lr]["best_val_acc"])

print("\nBest LR:")
print(f"{best_lr} with val_acc={results[best_lr]['best_val_acc']:.4f}")


# =========================================================
# W&B COMPARISON TABLE
# =========================================================

if wandb.run is not None:
    wandb.finish()

run = wandb.init(
    project="MPW-CNN",
    entity="MSE_DeLearn_SPR26",
    name="lr_tuning_MomentumNesterov_summary_depth12",
    reinit=True
)

comparison_table = wandb.Table(columns=[
    "lr",
    "best_val_acc",
    "best_train_acc",
    "best_epoch",
    "best_val_loss",
    "best_train_loss",
])

for lr, result in results.items():
    comparison_table.add_data(
        result["lr"],
        result["best_val_acc"],
        result["best_train_acc"],
        result["best_epoch"],
        result["best_val_loss"],
        result["best_train_loss"],
    )

wandb.log({
    "lr_comparison_table": comparison_table,
})

best_lr = max(results, key=lambda lr: results[lr]["best_val_acc"])

wandb.summary["best_lr"] = best_lr
wandb.summary["best_val_acc"] = results[best_lr]["best_val_acc"]
wandb.summary["best_train_acc"] = results[best_lr]["best_train_acc"]
wandb.summary["best_epoch"] = results[best_lr]["best_epoch"]
wandb.summary["best_val_loss"] = results[best_lr]["best_val_loss"]
wandb.summary["best_train_loss"] = results[best_lr]["best_train_loss"]

wandb.finish()

## Playing with the momentum

In [ ]:
from src.cnn_utils import get_dataset

train_dataset, val_dataset = get_dataset()

# check dataset loaded correctly
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

In [ ]:
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import wandb
from src.cnn_utils import *

# ---------------------------------------------------------
# config
# ---------------------------------------------------------
model_cfg = ModelConfig(depth=12)
wandb_cfg = WandbConfig(use_wandb=True)

device = get_device()

momentums = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0] # testing different momentum values from 0 (no momentum) to 1 (full momentum)
batch_size = 64 # default value
epochs = 50
lr = 0.01 # best learning rate from previous tuning
weight_decay = 1e-4 # default value

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

criterion = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# momentum sweep
# ---------------------------------------------------------

all_results = {} # to store results for each momentum value

wandb_login_if_needed(wandb_cfg)

for m in momentums:
    print(f"\n==== Training with momentum = {m} ====\n")

    # new model and optimizer for each momentum value
    model = build_model(model_cfg).to(device)
    num_parameters = get_num_parameters(model)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=m,
        weight_decay=weight_decay,
    )

    run = init_wandb_run(
        cfg=wandb_cfg,
        model=model,
        run_name=f"Momentum_{m}_depth{model_cfg.depth}_lr{lr}_e{epochs}",
        config_dict={
            "optimizer": "Momentum_SGD",
            "momentum": m,
            "lr": lr,
            "weight_decay": weight_decay,
            "depth": model_cfg.depth,
            "batch_size": batch_size,
            "epochs": epochs,
            "num_parameters": num_parameters,
            "device": str(device),
        },
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time_sec": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    best_train_acc = 0.0
    best_train_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    # ---------------------------
    # training loop
    # ---------------------------
    for epoch in range(epochs):

        # TRAIN
        model.train_model()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        start_time = time.time()

        for x, y in train_loader:
            x, y = x.to(device,non_blocking=True), y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            batch_n = y.size(0)
            train_loss_sum += loss.item() * batch_n
            train_correct += (logits.argmax(dim=1) == y).sum().item()
            train_total += batch_n

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # VALIDATION
        model.eval()
        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

                logits = model(x)
                loss = criterion(logits, y)

                batch_n = y.size(0)
                val_loss_sum += loss.item() * batch_n
                val_correct += (logits.argmax(dim=1) == y).sum().item()
                val_total += batch_n

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        epoch_time = time.time() - start_time

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["epoch_time_sec"].append(epoch_time)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_model_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        if run is not None:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
                "epoch_time_sec": epoch_time,
                "momentum": m,
                "best_val_accuracy_so_far": best_val_acc,
            })

    # restore best model state for this momentum run
    model.load_state_dict(best_model_state)

    avg_epoch_time = sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"])

    all_results[m] = {
        "momentum": m,
        "depth": model_cfg.depth,
        "num_parameters": num_parameters,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_train_acc": best_train_acc,
        "best_train_loss": best_train_loss,
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "avg_epoch_time_sec": avg_epoch_time,
    }

    if run is not None:
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_accuracy"] = best_val_acc
        wandb.summary["best_val_loss"] = best_val_loss
        wandb.summary["best_train_accuracy_at_best_val"] = best_train_acc
        wandb.summary["best_train_loss_at_best_val"] = best_train_loss
        wandb.summary["final_train_accuracy"] = history["train_acc"][-1]
        wandb.summary["final_val_accuracy"] = history["val_acc"][-1]
        wandb.summary["avg_epoch_time_sec"] = avg_epoch_time
        wandb.finish()

# =========================================================
# OPTIONAL: W&B SUMMARY RUN (MOMENTUM COMPARISON)
# =========================================================

if wandb_cfg.use_wandb and wandb_cfg.mode != "disabled":
    best_momentum = max(all_results.keys(), key=lambda k: all_results[k]["best_val_acc"])

    run = wandb.init(
        project=wandb_cfg.project,
        entity=wandb_cfg.entity,
        mode=wandb_cfg.mode,
        name=f"momentum_comparison_depth{model_cfg.depth}",
        reinit=True,
        settings=wandb.Settings(init_timeout=wandb_cfg.init_timeout),
    )

    comparison_table = wandb.Table(columns=[
        "momentum",
        "depth",
        "num_parameters",
        "best_val_acc",
        "best_val_loss",
        "best_epoch",
        "best_train_acc",
        "best_train_loss",
        "final_train_acc",
        "final_val_acc",
        "avg_epoch_time_sec",
    ])

    for momentum_value, result in all_results.items():
        comparison_table.add_data(
            result["momentum"],
            result["depth"],
            result["num_parameters"],
            result["best_val_acc"],
            result["best_val_loss"],
            result["best_epoch"],
            result["best_train_acc"],
            result["best_train_loss"],
            result["final_train_acc"],
            result["final_val_acc"],
            result["avg_epoch_time_sec"],
        )

    wandb.log({
        "momentum_comparison_table": comparison_table,
        "best_momentum": best_momentum,
        "best_momentum_val_acc": all_results[best_momentum]["best_val_acc"],
        "depth": model_cfg.depth,
    })

    wandb.summary["best_momentum"] = best_momentum
    wandb.summary["best_momentum_val_acc"] = all_results[best_momentum]["best_val_acc"]
    wandb.summary["depth"] = model_cfg.depth

    wandb.finish()


# ---------------------------------------------------------
# optional console summary
# ---------------------------------------------------------
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

for m, result in all_results.items():
    print(
        f"Momentum={result['momentum']:.1f} | "
        f"depth={result['depth']} | "
        f"params={result['num_parameters']:,} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"final_train_acc={result['final_train_acc']:.4f} | "
        f"final_val_acc={result['final_val_acc']:.4f} | "
        f"avg_epoch_time={result['avg_epoch_time_sec']:.2f}s"
    )

best_momentum = max(all_results.keys(), key=lambda k: all_results[k]["best_val_acc"])
print("\nBest momentum based on validation accuracy:")
print(f"Momentum = {best_momentum}")
print(f"Best validation accuracy = {all_results[best_momentum]['best_val_acc']:.4f}")

## Playing with the batchsize
using new configurationsetup

In [ ]:
import sys
from pathlib import Path
# fixing imports for provided_sources/ modules
sys.path.append(str(Path().resolve().parents[1]))

import copy
import pprint

import torch
import torch.nn as nn
import src.cnn.cnn_paths as paths
import src.cnn.cnn_models as cnn_models
from src.cnn.configs import ExperimentConfig
from src.cnn.cnn_registry import build_model
from src.cnn.cnn_utils import (
    load_datasets,
    print_dataset_info,
    make_train_loader,
    make_eval_loader,
    get_device,
    get_num_parameters,
    train_and_evaluate_model,
    evaluate_model_on_loader, run_experiment,
)

In [ ]:
# batch sizes to test
batch_sizes = [4, 1024, 2048]

all_results = {}

for bs in batch_sizes:
    print(f"\n==== Training with batch_size = {bs} ====\n")

    # see configs.py
    cfg = ExperimentConfig()

    # =========================================================
    # DATASET CONFIG
    # =========================================================
    cfg.dataset.dataset_dir = paths.DATASET_DIR # see cnn_paths.py
    cfg.dataset.train_subdir = "train"
    cfg.dataset.val_subdir = "validate"
    cfg.dataset.test_subdir = None   # or "test" if you have it
    cfg.dataset.image_size = 224

    # leave transforms as None -> default Resize + ToTensor pipeline
    cfg.dataset.train_transform = None
    cfg.dataset.eval_transform = None

    # optional normalization
    cfg.dataset.normalize_mean = None
    cfg.dataset.normalize_std = None


    # =========================================================
    # MODEL CONFIG
    # IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
    # use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
    # =========================================================
    cfg.model.name = "depth_cnn"
    cfg.model.kwargs = {
        "depth": 12,
        "in_channels": 3,
        "num_classes": 10,
        "base_channels": 32,
        "max_channels": 256,
        "dropout_conv": 0.0,
        "dropout_fc": 0.5,
        "inputsize": 224,
    }


    # =========================================================
    # DATALOADER CONFIG
    # =========================================================
    cfg.loader.batch_size = bs
    cfg.loader.num_workers = 0
    cfg.loader.pin_memory = True
    cfg.loader.train_shuffle = True
    cfg.loader.eval_shuffle = False
    cfg.loader.drop_last_train = False
    cfg.loader.drop_last_eval = False


    # =========================================================
    # LOSS CONFIG
    # =========================================================
    cfg.loss.cls = nn.CrossEntropyLoss
    cfg.loss.kwargs = {}


    # =========================================================
    # OPTIMIZER CONFIG
    # =========================================================
    cfg.optimizer.cls = torch.optim.SGD
    cfg.optimizer.kwargs = {
        "lr": 0.01, # from previous testings
        "momentum": 0.9, # default value for momentum
        "weight_decay": 1e-4, # deafault value
    }


    # =========================================================
    # SCHEDULER CONFIG
    # Example: Reduce LR when validation loss plateaus
    # =========================================================
    cfg.scheduler.cls = None
    cfg.scheduler.kwargs = {
        # "mode": "min",
        # "factor": 0.5,
        # "patience": 2,
    }
    cfg.scheduler.step_metric = None


    # =========================================================
    # TRAIN CONFIG
    # =========================================================
    cfg.train.epochs = 50
    cfg.train.device = str(get_device("auto"))
    cfg.train.non_blocking = True
    cfg.train.use_amp = (cfg.train.device == "cuda") # suggestion from ChatGPT for large batch sizes to save memory and speed up training
    cfg.train.grad_clip_norm = None
    cfg.train.best_metric = "val/accuracy"
    cfg.train.best_mode = "max"
    cfg.train.seed = 42


    # =========================================================
    # W&B CONFIG
    # =========================================================
    cfg.wandb.enabled = True
    cfg.wandb.project = "MPW-CNN"
    cfg.wandb.entity = "MSE_DeLearn_SPR26"
    cfg.wandb.mode = "online"   # "online", "offline", or "disabled" for no logging

    # NOTE: use meaningful names for runs, so the difference is clear
    cfg.wandb.run_name = f"BatchSize_{bs}_depth{cfg.model.kwargs['depth']}_momentum{cfg.optimizer.kwargs['momentum']}_lr{cfg.optimizer.kwargs['lr']}_e{cfg.train.epochs}"

    # NOTE: use meaningful grouping, for example, by model or by task.
    # E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
    cfg.wandb.group = "Batchsize_testing"
    cfg.wandb.job_type = "train"

    # NOTE: use meaningful tags to filter runs in UI
    cfg.wandb.tags = ["deep_cnn", "batch_size"]
    cfg.wandb.notes = "Full config live run"

    cfg.wandb.log_epoch_metrics = True
    cfg.wandb.log_every_n_epochs = 1

    cfg.wandb.metric_allowlist = {
        "train/loss",
        "train/accuracy",
        "val/loss",
        "val/accuracy",
        "gap/accuracy",
        "gap/loss",
        "lr",
    }

    cfg.wandb.summary_allowlist = {
        "best_epoch",
        "best_metric_name",
        "best_metric_value",
        "train_final/loss",
        "train_final/accuracy",
        "val_final/loss",
        "val_final/accuracy",
    }

    cfg.wandb.watch_model = False
    cfg.wandb.watch_log = "all"
    cfg.wandb.watch_log_freq = 100

    # ---------------------------
    # RUN
    # ---------------------------
    # with high batchsizes GPU memory may not be sufficient, so we catch OOM errors and skip those runs
    try:
        model, history, result = run_experiment(cfg)
        all_results[bs] = copy.deepcopy(result)

        # give some time for GPU memory to be freed before next run
        del model, history, result
        torch.cuda.empty_cache()

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OOM at batch size {bs}, skipping...")
            torch.cuda.empty_cache()
            break # if we get OOM at a certain batch size, it's likely that all larger batch sizes will also be OOM, so we can break the loop early
        else:
            raise

# ---------------------------------------------------------
# summary
# ---------------------------------------------------------
print("\n" + "="*80)
print("SUMMARY (Batch Size Comparison)")
print("="*80)

for bs, res in all_results.items():
    print(
        f"bs={bs:>3} | "
        f"best_val_acc={res['best_metric_value']:.4f} | "
        f"best_epoch={res['best_epoch']}"
    )

